<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>Module 2.3</b>  
  <span style="color:#9ca3af;">Chunking Best Practices</span>
</div>


## Optimal Chunk Sizes by Use Case
| Use Case | Token Range | Why |
|---|---|---|
| Factoid queries | 128–256 | Precise short answers |
| General RAG | 256–512 | Balance precision & context |
| Complex analysis | 512–1024 | Long reasoning chains |

## Key Rules
- **Overlap**: 10–20 % of chunk size (prevents answer splitting at boundaries)
- **Context preservation**: use `add_start_index=True` to track position
- **Tables**: keep in one chunk; do NOT split across rows

### Course alignment and free-first stack

- Covers: Chunk-size experiments, overlap trade-offs, and table-aware chunk handling.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.documents import Document
import time

DOCUMENT = """
Neural networks are computing systems inspired by biological neural networks that
constitute animal brains. They consist of layers of interconnected nodes or neurons
that process information using connectionist approaches to computation.

Deep learning is part of a broader family of machine learning methods based on
artificial neural networks with representation learning. Learning can be supervised,
semi-supervised or unsupervised.

Convolutional neural networks (CNNs) are a class of deep neural networks, most commonly
applied to analyse visual imagery. They use a variation of multilayer perceptrons designed
to require minimal preprocessing.

Recurrent neural networks (RNNs) are a class of artificial neural networks where
connections between nodes form a directed graph along a temporal sequence, exhibiting
temporal dynamic behaviour.
"""

QUERY = "What are convolutional neural networks?"
"""
Experiment: evaluate how chunk_size affects retrieval quality.
"""

embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))
results    = []

for chunk_size, overlap in [(128, 20), (256, 50), (512, 100)]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap, add_start_index=True
    )
    docs = splitter.create_documents([DOCUMENT])
    vs   = Chroma.from_documents(docs, embeddings,
                                 collection_name=f"test_{chunk_size}")
    retrieved = vs.similarity_search(QUERY, k=2)

    print(f"chunk_size={chunk_size:4d} | overlap={overlap:3d} "
          f"| chunks={len(docs):2d} | best_chunk_len={len(retrieved[0].page_content)}")
    print(f"  ↳ {retrieved[0].page_content[:120].strip()}\n")
    results.append({"chunk_size": chunk_size, "overlap": overlap,
                    "n_chunks": len(docs), "retrieved": retrieved[0].page_content})


## Handling Tables
Tables should stay intact. Use `PDFPlumber` for extraction and keep each table as a single `Document`.

In [ ]:
# ── Table preservation pattern ────────────────────────────────────────────────
from langchain_core.documents import Document

def table_to_document(table_rows: list[list], headers: list[str], source: str) -> Document:
    """Convert a parsed table into a single Document that won't be split."""
    header_str = " | ".join(headers)
    row_strs   = ["\n" + " | ".join(str(c) for c in row) for row in table_rows]
    content    = f"TABLE: {header_str}" + "".join(row_strs)
    return Document(page_content=content, metadata={"source": source, "type": "table"})

# Example
headers = ["Model", "Params", "MMLU Score"]
rows    = [["GPT-4o", "?", "87.5"], ["Claude 3.5", "?", "88.0"], ["Gemini 1.5", "?", "85.9"]]
table_doc = table_to_document(rows, headers, "model_comparison.pdf")
print(table_doc.page_content)
